In [1]:

import pandas as pd

file_path = r"C:\MAHAD\Master in Quant\Subjects\Term 3\Machine Learning\Assignment\Data\SP500_Daily_Data.xlsx"

df = pd.read_excel(file_path)

# Basic inspection
print(df.head())
print(df.tail())
print(df.shape)
print(df.info())

# Check missing values
print(df.isnull().sum())

# Convert date and sort
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Check date range
print("Start date:", df['date'].min())
print("End date:", df['date'].max())




        date     open     high        low    close  adjusted_close      volume
0 2000-01-03  1469.25  1478.00  1438.3600  1455.22         1455.22   931800000
1 2000-01-04  1455.22  1455.22  1397.4301  1399.42         1399.42  1009000000
2 2000-01-05  1399.42  1413.27  1377.6801  1402.11         1402.11  1085500000
3 2000-01-06  1402.11  1411.90  1392.1000  1403.45         1403.45  1092300000
4 2000-01-07  1403.45  1441.47  1400.7300  1441.47         1441.47  1225200000
           date       open       high        low      close  adjusted_close  \
6534 2025-12-24  6904.9102  6937.3198  6904.9102  6932.0498       6932.0498   
6535 2025-12-26  6936.0200  6945.7700  6921.6001  6929.9399       6929.9399   
6536 2025-12-29  6903.6001  6920.2100  6888.7598  6905.7402       6905.7402   
6537 2025-12-30  6900.4399  6913.2500  6893.4702  6896.2402       6896.2402   
6538 2025-12-31  6898.8198  6901.4199  6844.5498  6845.5000       6845.5000   

          volume  
6534  1798270000  
6535  2586550

In [3]:
df['daily_return'] = df['adjusted_close'].pct_change()

# Calculate next-day return
df['next_day_return'] = df['daily_return'].shift(-1)

# Create target variable
df['target'] = (df['next_day_return'] > 0).astype(int)

# Remove rows created by pct_change and shift
df = df.dropna().reset_index(drop=True)

# Check target distribution
print(df[['date', 'adjusted_close', 'daily_return', 'next_day_return', 'target']].head())
print(df['target'].value_counts())
print(df['target'].value_counts(normalize=True))

        date  adjusted_close  daily_return  next_day_return  target
0 2000-01-04         1399.42     -0.038345         0.001922       1
1 2000-01-05         1402.11      0.001922         0.000956       1
2 2000-01-06         1403.45      0.000956         0.027090       1
3 2000-01-07         1441.47      0.027090         0.011190       1
4 2000-01-10         1457.60      0.011190        -0.013062       0
target
1    3513
0    3024
Name: count, dtype: int64
target
1    0.537402
0    0.462598
Name: proportion, dtype: float64


In [7]:
# Feature engineering

# Lagged returns
df['return_lag_1'] = df['daily_return'].shift(1)
df['return_lag_2'] = df['daily_return'].shift(2)
df['return_lag_3'] = df['daily_return'].shift(3)
df['return_lag_5'] = df['daily_return'].shift(5)

# Moving averages relative to adjusted close
df['ma_5'] = df['adjusted_close'].rolling(window=5).mean()
df['ma_20'] = df['adjusted_close'].rolling(window=20).mean()

df['ma_5_ratio'] = df['adjusted_close'] / df['ma_5'] - 1
df['ma_20_ratio'] = df['adjusted_close'] / df['ma_20'] - 1

# Rolling volatility
df['volatility_5'] = df['daily_return'].rolling(window=5).std()
df['volatility_20'] = df['daily_return'].rolling(window=20).std()

# Momentum
df['momentum_5'] = df['adjusted_close'] / df['adjusted_close'].shift(5) - 1
df['momentum_10'] = df['adjusted_close'] / df['adjusted_close'].shift(10) - 1

# Intraday price features
df['high_low_range'] = (df['high'] - df['low']) / df['close']
df['open_close_return'] = (df['close'] - df['open']) / df['open']

# Volume features
df['volume_change'] = df['volume'].pct_change()
df['volume_ma_5'] = df['volume'].rolling(window=5).mean()
df['volume_ma_5_ratio'] = df['volume'] / df['volume_ma_5'] - 1

# Drop rows with missing values created by rolling windows and lags
df = df.dropna().reset_index(drop=True)

# Check final dataset
print(df.head())
print(df.shape)
print(df.isnull().sum())

        date     open     high      low    close  adjusted_close      volume  \
0 2000-02-01  1394.46  1412.49  1384.79  1409.28         1409.28   981000000   
1 2000-02-02  1409.28  1420.61  1403.49  1409.12         1409.12  1038600000   
2 2000-02-03  1409.12  1425.78  1398.52  1424.97         1424.97  1146500000   
3 2000-02-04  1424.97  1435.91  1420.63  1424.37         1424.37  1045100000   
4 2000-02-07  1424.37  1427.15  1413.33  1424.24         1424.24   918100000   

   daily_return  next_day_return  target  ...  ma_20_ratio  volatility_5  \
0      0.010628        -0.000114       0  ...    -0.009842      0.019596   
1     -0.000114         0.011248       1  ...    -0.010292      0.019458   
2      0.011248        -0.000421       0  ...     0.000038      0.019704   
3     -0.000421        -0.000091       0  ...    -0.001117      0.010512   
4     -0.000091         0.012273       1  ...    -0.000604      0.006111   

   volatility_20  momentum_5  momentum_10  high_low_range  ope

In [22]:
# Define predictor features
import numpy as np

# Define predictor features again
feature_cols = [
    'return_lag_1',
    'return_lag_2',
    'return_lag_3',
    'return_lag_5',
    'ma_5_ratio',
    'ma_20_ratio',
    'volatility_5',
    'volatility_20',
    'momentum_5',
    'momentum_10',
    'high_low_range',
    'open_close_return',
    'volume_change',
    'volume_ma_5_ratio'
]

# Check which columns have infinity values
inf_counts = np.isinf(df[feature_cols]).sum()
print("Infinity values by column:")
print(inf_counts[inf_counts > 0])

# Replace infinity with NaN
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)

# Drop rows with NaN or infinity-created missing values
df = df.dropna(subset=feature_cols + ['target']).reset_index(drop=True)

# Confirm problem is solved
print("Remaining missing values:")
print(df[feature_cols].isnull().sum())

print("Final dataset shape:", df.shape)

X = df[feature_cols]
y = df['target']

# Time-based train/test split: 80% train, 20% test
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

train_dates = df['date'].iloc[:split_index]
test_dates = df['date'].iloc[split_index:]

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("Training period:", train_dates.min(), "to", train_dates.max())
print("Test period:", test_dates.min(), "to", test_dates.max())

print("Training target distribution:")
print(y_train.value_counts(normalize=True))

print("Test target distribution:")
print(y_test.value_counts(normalize=True))

Infinity values by column:
volume_change    1
dtype: int64
Remaining missing values:
return_lag_1         0
return_lag_2         0
return_lag_3         0
return_lag_5         0
ma_5_ratio           0
ma_20_ratio          0
volatility_5         0
volatility_20        0
momentum_5           0
momentum_10          0
high_low_range       0
open_close_return    0
volume_change        0
volume_ma_5_ratio    0
dtype: int64
Final dataset shape: (6517, 27)
Training set: (5213, 14)
Test set: (1304, 14)
Training period: 2000-02-01 00:00:00 to 2020-10-19 00:00:00
Test period: 2020-10-20 00:00:00 to 2025-12-30 00:00:00
Training target distribution:
target
1    0.537311
0    0.462689
Name: proportion, dtype: float64
Test target distribution:
target
1    0.537577
0    0.462423
Name: proportion, dtype: float64


In [24]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

# Create baseline model: always predicts the majority class
baseline_model = DummyClassifier(strategy='most_frequent')

# Fit baseline on training data only
baseline_model.fit(X_train, y_train)

# Predict on test data
baseline_pred = baseline_model.predict(X_test)

# Get predicted probabilities for class 1
baseline_proba = baseline_model.predict_proba(X_test)[:, 1]

# Evaluate baseline
baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_auc = roc_auc_score(y_test, baseline_proba)
baseline_cm = confusion_matrix(y_test, baseline_pred)

# Print results
print("Baseline Accuracy:", baseline_accuracy)
print("Baseline ROC-AUC:", baseline_auc)

print("\nBaseline Confusion Matrix:")
print(baseline_cm)

print("\nClassification Report:")
print(classification_report(y_test, baseline_pred, zero_division=0))


Baseline Accuracy: 0.5375766871165644
Baseline ROC-AUC: 0.5

Baseline Confusion Matrix:
[[  0 603]
 [  0 701]]

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       603
           1       0.54      1.00      0.70       701

    accuracy                           0.54      1304
   macro avg       0.27      0.50      0.35      1304
weighted avg       0.29      0.54      0.38      1304



In [26]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

# Create leakage-safe pipeline
logistic_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

# Time-series cross-validation on training data only
tscv = TimeSeriesSplit(n_splits=5)

cv_auc_scores = cross_val_score(
    logistic_pipeline,
    X_train,
    y_train,
    cv=tscv,
    scoring='roc_auc'
)

print("Logistic Regression CV ROC-AUC scores:", cv_auc_scores)
print("Mean CV ROC-AUC:", cv_auc_scores.mean())

# Fit model on full training set
logistic_pipeline.fit(X_train, y_train)

# Predict on test set
logistic_pred = logistic_pipeline.predict(X_test)
logistic_proba = logistic_pipeline.predict_proba(X_test)[:, 1]

# Evaluate test performance
logistic_accuracy = accuracy_score(y_test, logistic_pred)
logistic_auc = roc_auc_score(y_test, logistic_proba)
logistic_cm = confusion_matrix(y_test, logistic_pred)

print("\nTest Accuracy:", logistic_accuracy)
print("Test ROC-AUC:", logistic_auc)

print("\nConfusion Matrix:")
print(logistic_cm)

print("\nClassification Report:")
print(classification_report(y_test, logistic_pred, zero_division=0))

Logistic Regression CV ROC-AUC scores: [0.54140962 0.537431   0.48194839 0.51728961 0.52396891]
Mean CV ROC-AUC: 0.520409505993191

Test Accuracy: 0.5299079754601227
Test ROC-AUC: 0.502750157912293

Confusion Matrix:
[[ 86 517]
 [ 96 605]]

Classification Report:
              precision    recall  f1-score   support

           0       0.47      0.14      0.22       603
           1       0.54      0.86      0.66       701

    accuracy                           0.53      1304
   macro avg       0.51      0.50      0.44      1304
weighted avg       0.51      0.53      0.46      1304



In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1
)

# Time-series cross-validation on training data only
tscv = TimeSeriesSplit(n_splits=5)

rf_cv_auc_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=tscv,
    scoring='roc_auc'
)

print("Random Forest CV ROC-AUC scores:", rf_cv_auc_scores)
print("Mean CV ROC-AUC:", rf_cv_auc_scores.mean())

# Fit model on full training set
rf_model.fit(X_train, y_train)

# Predict on test set
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

# Evaluate test performance
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)
rf_cm = confusion_matrix(y_test, rf_pred)

print("\nTest Accuracy:", rf_accuracy)
print("Test ROC-AUC:", rf_auc)

print("\nConfusion Matrix:")
print(rf_cm)

print("\nClassification Report:")
print(classification_report(y_test, rf_pred, zero_division=0))

Random Forest CV ROC-AUC scores: [0.52686044 0.55147464 0.47643337 0.51759275 0.51286051]
Mean CV ROC-AUC: 0.5170443403920175

Test Accuracy: 0.5368098159509203
Test ROC-AUC: 0.5054163325076946

Confusion Matrix:
[[ 73 530]
 [ 74 627]]

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.12      0.19       603
           1       0.54      0.89      0.67       701

    accuracy                           0.54      1304
   macro avg       0.52      0.51      0.43      1304
weighted avg       0.52      0.54      0.45      1304

